# House Prices Iowa

This project predicts house sale prices in Ames, Iowa, using 79 features that describe almost every aspect of a residential home (size, quality, location, age, garage, basement, and more). It is the classic regression competition on Kaggle, with a large number of features and a skewed target.

## Approach
1. Load the data and explore the target and missing values
2. Handle missing values (they often have a meaning, e.g. "no garage")
3. Log-transform the skewed target, since the competition scores on the log scale
4. Encode categorical features and select a feature set
5. Train a baseline regression model and evaluate it with a validation split
6. Generate predictions and create the submission file
7. Submit to Kaggle and record the score

In [2]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (starts with KGAT_) and press Enter: ")

!pip install -q -U kaggle
!kaggle competitions download -c house-prices-advanced-regression-techniques
!unzip -oq house-prices-advanced-regression-techniques.zip

Paste your Kaggle API token (starts with KGAT_) and press Enter: ··········
100% 199k/199k [00:00<00:00, 82.0MB/s]



In [3]:
import pandas as pd
import numpy as np

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [4]:
train.shape, test.shape


((1460, 81), (1459, 80))

In [5]:
train.sample(1)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1457,1458,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,GdPrv,Shed,2500,5,2010,WD,Normal,266500


In [6]:
test.sample(1)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
603,2064,20,RL,102.0,9373,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,5,2008,WD,Normal


In [7]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   object 
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   object 
 6   Alley          107 non-null    object 
 7   LotShape       1459 non-null   object 
 8   LandContour    1459 non-null   object 
 9   Utilities      1457 non-null   object 
 10  LotConfig      1459 non-null   object 
 11  LandSlope      1459 non-null   object 
 12  Neighborhood   1459 non-null   object 
 13  Condition1     1459 non-null   object 
 14  Condition2     1459 non-null   object 
 15  BldgType       1459 non-null   object 
 16  HouseStyle     1459 non-null   object 
 17  OverallQual    1459 non-null   int64  
 18  OverallC

In [8]:
train["SalePrice"].describe()

,SalePrice
count,1460.000000
mean,180921.195890
std,79442.502883
min,34900.000000
25%,129975.000000
50%,163000.000000
75%,214000.000000
max,755000.000000


In [9]:
train.isnull().sum().sort_values(ascending=False).head(15)

,0
PoolQC,1453
MiscFeature,1406
Alley,1369
Fence,1179
MasVnrType,872
FireplaceQu,690
LotFrontage,259
GarageQual,81
GarageFinish,81
GarageType,81


In [10]:
train["PoolQC"] = train["PoolQC"].fillna("None")

In [11]:
train["MiscFeature"] = train["MiscFeature"].fillna("None")

In [12]:
train["Alley"] = train["Alley"].fillna("None")
train["Fence"] = train["Fence"].fillna("None")
train["FireplaceQu"] = train["FireplaceQu"].fillna("None")
train["MasVnrType"] = train["MasVnrType"].fillna("None")
train["GarageType"] = train["GarageType"].fillna("None")
train["GarageFinish"] = train["GarageFinish"].fillna("None")
train["GarageQual"] = train["GarageQual"].fillna("None")
train["GarageCond"] = train["GarageCond"].fillna("None")
train["BsmtQual"] = train["BsmtQual"].fillna("None")
train["BsmtCond"] = train["BsmtCond"].fillna("None")
train["BsmtExposure"] = train["BsmtExposure"].fillna("None")
train["BsmtFinType1"] = train["BsmtFinType1"].fillna("None")
train["BsmtFinType2"] = train["BsmtFinType2"].fillna("None")

In [13]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          1460 non-null   object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [14]:
train[["PoolQC", "GarageType", "BsmtQual"]].isnull().sum()

,0
PoolQC,0
GarageType,0
BsmtQual,0


In [15]:
test["PoolQC"] = test["PoolQC"].fillna("None")
test["MiscFeature"] = test["MiscFeature"].fillna("None")
test["Alley"] = test["Alley"].fillna("None")
test["Fence"] = test["Fence"].fillna("None")
test["FireplaceQu"] = test["FireplaceQu"].fillna("None")
test["MasVnrType"] = test["MasVnrType"].fillna("None")
test["GarageType"] = test["GarageType"].fillna("None")
test["GarageFinish"] = test["GarageFinish"].fillna("None")
test["GarageQual"] = test["GarageQual"].fillna("None")
test["GarageCond"] = test["GarageCond"].fillna("None")
test["BsmtQual"] = test["BsmtQual"].fillna("None")
test["BsmtCond"] = test["BsmtCond"].fillna("None")
test["BsmtExposure"] = test["BsmtExposure"].fillna("None")
test["BsmtFinType1"] = test["BsmtFinType1"].fillna("None")
test["BsmtFinType2"] = test["BsmtFinType2"].fillna("None")

In [16]:
test[["PoolQC", "GarageType", "BsmtQual"]].isnull().sum()

,0
PoolQC,0
GarageType,0
BsmtQual,0


In [17]:
train["GarageYrBlt"] = train["GarageYrBlt"].fillna(0)
train["GarageArea"] = train["GarageArea"].fillna(0)
train["GarageCars"] = train["GarageCars"].fillna(0)
train["MasVnrArea"] = train["MasVnrArea"].fillna(0)
train["BsmtFinSF1"] = train["BsmtFinSF1"].fillna(0)
train["BsmtFinSF2"] = train["BsmtFinSF2"].fillna(0)
train["BsmtUnfSF"] = train["BsmtUnfSF"].fillna(0)
train["TotalBsmtSF"] = train["TotalBsmtSF"].fillna(0)
train["BsmtFullBath"] = train["BsmtFullBath"].fillna(0)
train["BsmtHalfBath"] = train["BsmtHalfBath"].fillna(0)

In [18]:
train[["GarageYrBlt", "TotalBsmtSF", "BsmtFullBath"]].isnull().sum()

,0
GarageYrBlt,0
TotalBsmtSF,0
BsmtFullBath,0


In [19]:
test["GarageYrBlt"] = test["GarageYrBlt"].fillna(0)
test["GarageArea"] = test["GarageArea"].fillna(0)
test["GarageCars"] = test["GarageCars"].fillna(0)
test["MasVnrArea"] = test["MasVnrArea"].fillna(0)
test["BsmtFinSF1"] = test["BsmtFinSF1"].fillna(0)
test["BsmtFinSF2"] = test["BsmtFinSF2"].fillna(0)
test["BsmtUnfSF"] = test["BsmtUnfSF"].fillna(0)
test["TotalBsmtSF"] = test["TotalBsmtSF"].fillna(0)
test["BsmtFullBath"] = test["BsmtFullBath"].fillna(0)
test["BsmtHalfBath"] = test["BsmtHalfBath"].fillna(0)

In [20]:
test[["GarageYrBlt", "TotalBsmtSF", "BsmtFullBath"]].isnull().sum()

,0
GarageYrBlt,0
TotalBsmtSF,0
BsmtFullBath,0


In [21]:
train["LotFrontage"] = train.groupby("Neighborhood")["LotFrontage"].transform(lambda s: s.fillna(s.median()))

In [22]:
train["LotFrontage"].isnull().sum()

np.int64(0)

In [23]:
test["LotFrontage"] = test.groupby("Neighborhood")["LotFrontage"].transform(lambda s: s.fillna(s.median()))

In [24]:
test["LotFrontage"].isnull().sum()

np.int64(0)

In [25]:
train.isnull().sum()[lambda s: s > 0]

,0
Electrical,1


In [26]:
test.isnull().sum()[lambda s: s > 0]

,0
MSZoning,4
Utilities,2
Exterior1st,1
Exterior2nd,1
KitchenQual,1
Functional,2
SaleType,1


In [27]:
train["Electrical"] = train["Electrical"].fillna(train["Electrical"].mode()[0])

In [28]:
test["MSZoning"] = test["MSZoning"].fillna(test["MSZoning"].mode()[0])
test["Utilities"] = test["Utilities"].fillna(test["Utilities"].mode()[0])
test["Exterior1st"] = test["Exterior1st"].fillna(test["Exterior1st"].mode()[0])
test["Exterior2nd"] = test["Exterior2nd"].fillna(test["Exterior2nd"].mode()[0])
test["KitchenQual"] = test["KitchenQual"].fillna(test["KitchenQual"].mode()[0])
test["Functional"] = test["Functional"].fillna(test["Functional"].mode()[0])
test["SaleType"] = test["SaleType"].fillna(test["SaleType"].mode()[0])

In [29]:
print(train.isnull().sum().sum())

0


In [30]:
print(test.isnull().sum().sum())

0


In [31]:
train["SalePriceLog"] = np.log1p(train["SalePrice"])

In [32]:
train["SalePrice"].skew(), train["SalePriceLog"].skew()

(np.float64(1.8828757597682129), np.float64(0.12134661989685333))

In [33]:
train = pd.get_dummies(train)

In [34]:
test= pd.get_dummies(test)

In [35]:
train, test = train.align(test, join="left", axis=1, fill_value=0)

In [36]:
train.shape, test.shape

((1460, 305), (1459, 305))

In [38]:
list(train.columns) == list(test.columns)

True

In [39]:
x = train.drop(columns=["Id", "SalePrice", "SalePriceLog"])
y = train["SalePriceLog"]

In [40]:
x.shape, y.shape

((1460, 302), (1460,))

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(x_train, y_train)

pred = model.predict(x_val)
rmse = mean_squared_error(y_val, pred) ** 0.5
print("Validation RMSE (log scale):", round(rmse, 4))

Validation RMSE (log scale): 0.1457


In [42]:
model.fit(x, y)

x_test = test.drop(columns=["Id", "SalePrice", "SalePriceLog"])
pred_log = model.predict(x_test)
pred = np.expm1(pred_log)

pd.DataFrame({"Id": test["Id"], "SalePrice": pred}).to_csv("submission.csv", index=False)

In [43]:
!kaggle competitions submit -c house-prices-advanced-regression-techniques -f submission.csv -m "RandomForest baseline"

100% 33.7k/33.7k [00:00<00:00, 158kB/s]
9 submissions remaining today.
Successfully submitted to House Prices - Advanced Regression Techniques

In [46]:
import pickle
model = RandomForestRegressor(n_estimators=150, max_depth=15, min_samples_leaf=3, random_state=42)
model.fit(x, y)

pickle.dump(model, open("house_model.pkl", "wb"))
print("size MB:", round(os.path.getsize("house_model.pkl") / 1e6, 1))

size MB: 5.5
